In [13]:
import pandas as pd
import numpy as np
import glob

POLICY_PATH = "/Users/aleksandar.stojanovic/Downloads/ai_system/ml_scripts/data/processed/policy_model_data.parquet"
WIN_PATH     = "/Users/aleksandar.stojanovic/Downloads/ai_system/ml_scripts/data/processed/win_model_data.parquet"
# === Load all parquet files in directory ===
files = glob.glob("/Users/aleksandar.stojanovic/Downloads/ai_system/ml_scripts/data/raw/*.parquet")
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)


# === Helper function to encode board state ===
def encode_board_state(board):
    symbol_map = {'X': 1, 'O': -1, ' ': 0}
    return [symbol_map.get(cell, 0) for cell in board]

# === Encode legal moves as binary vector ===
def encode_legal_moves(legal_moves):
    binary = [0] * 9
    for i in legal_moves:
        if 0 <= i < 9:
            binary[i] = 1
    return binary

# === Common processing: apply encoding ===
df["boardEncoded"] = df["boardState"].apply(encode_board_state)
df["playerEncoded"] = df["currentPlayer"].map({'X': 1, 'O': -1})
df["legalMovesEncoded"] = df["legalMoves"].apply(encode_legal_moves)

# === Filter 1: Policy model (only rows with strong label present) ===
policy_df = df[df["bestMoveStrong"].notnull()].copy()

# Features: board, player, (optionally legal moves)
policy_df["features"] = policy_df.apply(
    lambda row: row["boardEncoded"] + [row["playerEncoded"]],
    axis=1
)

# Target
policy_df["target"] = policy_df["bestMoveStrong"].astype(int)

# Save
policy_df[["features", "target"]].to_parquet(POLICY_PATH, index=False)


# Outcome encoding map
outcome_map = {'X_WON': 0, 'O_WON': 1, 'DRAW': 2}

# 1. Create empty column for encoded outcomes
df["outcomeEncoded"] = None

# 2. Assign final game outcome to every row in that session
for session_id, group in df.groupby("sessionId"):
    final_outcome = group["outcome"].dropna().iloc[-1] if group["outcome"].notnull().any() else None
    if final_outcome in outcome_map:
        df.loc[group.index, "outcomeEncoded"] = outcome_map[final_outcome]

# 3. Filter only rows with known outcomes (i.e., all moves from games that ended properly)
df = df[df["outcomeEncoded"].notnull()].copy()

# 4. Build features: board state (9), player (1), move number (1) = 11 total
df["features"] = df.apply(
    lambda row: row["boardEncoded"] + [row["playerEncoded"], row["moveNumber"]],
    axis=1
)
df["target"] = df["outcomeEncoded"].astype(int)

# 6. Save
df[["features", "target"]].to_parquet(WIN_PATH, index=False)


print("Preprocessing complete. Saved:")
print("- policy_model_data.parquet")
print("- win_model_data.parquet")


Preprocessing complete. Saved:
- policy_model_data.parquet
- win_model_data.parquet


In [14]:
from collections import Counter
import pandas as pd

# Load the processed win model dataset
win_df = pd.read_parquet("/Users/aleksandar.stojanovic/Downloads/ai_system/ml_scripts/data/processed/win_model_data.parquet")

# 1. Check row count
print(f"🧠 Total rows in win model data: {len(win_df)}")

# 2. Check for nulls
null_features = win_df["features"].isnull().sum()
null_target = win_df["target"].isnull().sum()
print(f"🧪 Nulls - features: {null_features}, target: {null_target}")

# 3. Distribution of target labels (0 = X_WON, 1 = O_WON, 2 = DRAW)
print("📊 Target label distribution:")
print(Counter(win_df["target"]))

# 4. Check length of feature vectors (should be 11: 9 board + player + moveNumber)
lengths = win_df["features"].apply(lambda x: len(x))
length_counts = lengths.value_counts()
print("📐 Feature length distribution:")
print(length_counts.sort_index())

# 5. Show sample rows
print("\n🔍 Sample rows:")
print(win_df.head(3))




🧠 Total rows in win model data: 42342
🧪 Nulls - features: 0, target: 0
📊 Target label distribution:
Counter({0: 25766, 1: 15478, 2: 1098})
📐 Feature length distribution:
features
11    42342
Name: count, dtype: int64

🔍 Sample rows:
                             features  target
0   [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]       1
1  [0, 0, 0, 0, 1, 0, 0, 0, 0, -1, 1]       1
2  [0, 0, 0, 0, 1, 0, 0, 0, -1, 1, 2]       1


In [73]:
import mlflow
print(mlflow.get_tracking_uri())


sqlite:///mlflow.db


In [15]:
# 1. Load full dataset
df_all = pd.read_parquet("/Users/aleksandar.stojanovic/Downloads/ai_system/ml_scripts/data/raw/raw2/dataset_v4.parquet")

# 2. Identify complete draw games
draw_ids = df_all[df_all["outcome"] == "DRAW"]["sessionId"].unique()
df_draws = df_all[df_all["sessionId"].isin(draw_ids)].copy()

# 3. Encode board + player
symbol_map = {"X": 1, "O": -1, " ": 0}
player_map = {"X": 1, "O": -1}
outcome_map = {"WIN_X": 0, "WIN_O": 1, "DRAW": 2}

df_draws["boardEncoded"] = df_draws["boardState"].apply(
    lambda board: [symbol_map.get(cell, 0) for cell in board]
)
df_draws["playerEncoded"] = df_draws["currentPlayer"].map(player_map)

# 4. Assign final game outcome to every row
df_draws["outcomeEncoded"] = None
for session_id, group in df_draws.groupby("sessionId"):
    final_outcome = group["outcome"].dropna().iloc[-1] if group["outcome"].notnull().any() else None
    if final_outcome in outcome_map:
        df_draws.loc[group.index, "outcomeEncoded"] = outcome_map[final_outcome]

# 5. Filter to valid rows
df_draws = df_draws[df_draws["outcomeEncoded"].notnull()].copy()

# 6. Build features
df_draws["features"] = df_draws.apply(
    lambda row: row["boardEncoded"] + [row["playerEncoded"], row["moveNumber"]],
    axis=1
)
df_draws["target"] = df_draws["outcomeEncoded"].astype(int)

# 7. Save to optional file
WIN_PATH     = "/Users/aleksandar.stojanovic/Downloads/ai_system/ml_scripts/data/processed/win_model_data_draw.parquet"

df_draws[["features", "target"]].to_parquet(WIN_PATH, index=False)
print(f"Saved {len(df_draws)} draw rows.")


Saved 1395 draw rows.


In [6]:
import os

print(os.path.abspath("mlflow.db"))
print("Exists:", os.path.exists("mlflow.db"))

print


/Users/aleksandar.stojanovic/Downloads/ai_system/ml_scripts/mlflow.db
Exists: True


In [18]:
import pandas as pd

df = pd.read_parquet(WIN_PATH)

labels = {0: "WIN_X", 1: "WIN_O", 2: "DRAW"}

counts = df["target"].value_counts().sort_index()
percents = df["target"].value_counts(normalize=True).sort_index() * 100

summary = pd.DataFrame({
    "count": counts,
    "percent": percents.round(2)
})

summary.index = summary.index.map(labels)

print(summary)
print("\nTOTAL:", counts.sum())


        count  percent
target                
DRAW     1395    100.0

TOTAL: 1395
